# Figure 3: Modeling cancer incidence as a function of age and TCR diversity

This notebook reproduces Figure 3 of the manuscript.

Panel A fits cancer incidence in males and females using a shared model in which sex differences enter only through sex-specific TCR diversity curves.

Panel B shows the model-implied relative cancer risk as a function of age and TCR diversity percentile.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.interpolate import CubicSpline, RectBivariateSpline
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
import scipy.stats as stats

from matplotlib.colors import LogNorm

%config InlineBackend.figure_format = "pdf"

plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14
plt.rcParams["axes.labelsize"] = 16

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

RANDOM_SEED = 1
rng = np.random.default_rng(RANDOM_SEED)

In [2]:
# Public TCR repertoire data hosted on Zenodo.
data_url = "https://zenodo.org/records/13993996/files/TCR_size_diversity_age_sex.csv?download=1"

df = (
    pd.read_csv(data_url)
    .rename(
        columns={
            "#Subject_ID": "subject_id",
            " age": "age",
            " sex": "sex",
            " size": "repertoire_size",
            " diversity": "tcr_diversity",
        }
    )
)

df["sex"] = df["sex"].str.strip().str.lower()

df["log_repertoire_size"] = np.log10(df["repertoire_size"])
df["log_tcr_diversity"] = np.log10(df["tcr_diversity"])

df.head()

,subject_id,age,sex,repertoire_size,tcr_diversity,log_repertoire_size,log_tcr_diversity
0,30000,63,male,508139,274815,5.705983,5.439040
1,30001,76,female,263716,167611,5.421136,5.224303
2,30002,70,female,519416,261836,5.715515,5.418029
3,30003,44,male,576294,383460,5.760644,5.583720
4,30004,47,male,505447,290642,5.703676,5.463358


In [3]:
def bootstrap_statistic(values, statistic=np.median, n_bootstrap=1001, rng=None):
    """
    Bootstrap uncertainty for a statistic.
    """
    if rng is None:
        rng = np.random.default_rng()

    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    boot = []
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        boot.append(statistic(sample))

    return statistic(values), np.std(boot)


def binned_summary(x, y, bins, interval=(0.16, 0.84), n_bootstrap=1001, rng=None):
    """
    Binned medians, bootstrap errors, and central intervals.
    """
    if rng is None:
        rng = np.random.default_rng()

    x = np.asarray(x)
    y = np.asarray(y)

    rows = []

    for lower, upper in bins:
        in_bin = (
            np.isfinite(x)
            & np.isfinite(y)
            & (x >= lower)
            & (x < upper)
        )

        x_bin = x[in_bin]
        y_bin = y[in_bin]

        if len(y_bin) <= 1:
            continue

        y_median, y_error = bootstrap_statistic(
            y_bin,
            statistic=np.median,
            n_bootstrap=n_bootstrap,
            rng=rng,
        )

        rows.append(
            {
                "age_median": np.median(x_bin),
                "value_median": y_median,
                "value_error": y_error,
                "interval_low": np.quantile(y_bin, interval[0]),
                "interval_high": np.quantile(y_bin, interval[1]),
                "n_subjects": len(y_bin),
            }
        )

    return pd.DataFrame(rows)

In [4]:
# Decade-wide bins used throughout the manuscript.
age_bin_lower = np.arange(9) * 10
age_bin_upper = age_bin_lower + 10
age_bin_upper[-1] = 100

age_bins = list(zip(age_bin_lower, age_bin_upper))

male = df["sex"] == "male"
female = df["sex"] == "female"

# Figure 3 model uses TCR diversity in linear units, not log units.
male_diversity = binned_summary(
    df.loc[male, "age"],
    df.loc[male, "tcr_diversity"],
    bins=age_bins,
    interval=(0.16, 0.84),
    n_bootstrap=1001,
    rng=rng,
)

female_diversity = binned_summary(
    df.loc[female, "age"],
    df.loc[female, "tcr_diversity"],
    bins=age_bins,
    interval=(0.16, 0.84),
    n_bootstrap=1001,
    rng=rng,
)

# Interpolate sex-specific median diversity curves and bootstrap errors.
male_diversity_spline = CubicSpline(
    male_diversity["age_median"],
    male_diversity["value_median"],
)

female_diversity_spline = CubicSpline(
    female_diversity["age_median"],
    female_diversity["value_median"],
)

male_diversity_error_spline = CubicSpline(
    male_diversity["age_median"],
    male_diversity["value_error"],
)

female_diversity_error_spline = CubicSpline(
    female_diversity["age_median"],
    female_diversity["value_error"],
)

male_diversity

,age_median,value_median,value_error,interval_low,interval_high,n_subjects
0,18.0,414378.5,11464.030763,293724.68,566688.48,272
1,25.0,376585.0,6159.691348,277155.80,503189.52,754
2,36.0,348369.5,2233.175953,251324.24,468973.68,2564
3,45.0,321370.5,2072.724085,224520.40,431129.20,3642
4,54.0,288731.0,2040.323711,199978.84,401413.16,3777
5,63.0,255972.0,2684.373662,166791.16,362106.88,2475
6,73.0,212950.0,3706.804174,132769.40,311519.84,779
7,82.0,147777.5,10893.953847,97363.36,247111.44,108


In [5]:
# Age grid for SEER incidence data.
# These are 5-year age-bin midpoints.
incidence_ages = np.arange(22, 87, 5)

# Female cancer incidence per 100,000 person-years.
female_incidence_table = [
    [5, 32.7, 0.2, 32.4, 33.1, 36887, 112659055],
    [6, 48.7, 0.2, 48.3, 49.1, 55432, 113753076],
    [7, 68.7, 0.2, 68.2, 69.2, 77869, 113384626],
    [8, 91.6, 0.3, 91.0, 92.2, 104087, 113629699],
    [9, 127.9, 0.3, 127.2, 128.5, 146191, 114327397],
    [10, 187.7, 0.4, 186.9, 188.5, 213821, 113931690],
    [11, 283.9, 0.5, 282.9, 284.9, 312969, 110227905],
    [12, 393.0, 0.6, 391.8, 394.3, 396497, 100881602],
    [13, 554.3, 0.8, 552.7, 555.8, 473237, 85382458],
    [14, 787.1, 1.1, 785.0, 789.2, 544951, 69234131],
    [15, 1021.9, 1.4, 1019.2, 1024.5, 565020, 55293719],
    [16, 1263.8, 1.7, 1260.5, 1267.1, 550097, 43527586],
    [17, 1411.6, 2.1, 1407.6, 1415.7, 470937, 33361284],
]

# Male cancer incidence per 100,000 person-years.
male_incidence_table = [
    [5, 24.8, 0.1, 24.5, 25.1, 29470, 118858713],
    [6, 35.4, 0.2, 35.1, 35.8, 41487, 117061368],
    [7, 53.0, 0.2, 52.6, 53.4, 61141, 115343973],
    [8, 81.3, 0.3, 80.8, 81.8, 92962, 114343926],
    [9, 132.5, 0.3, 131.8, 133.2, 150708, 113735769],
    [10, 227.5, 0.5, 226.7, 228.4, 255684, 112365665],
    [11, 393.9, 0.6, 392.7, 395.1, 422460, 107260632],
    [12, 615.9, 0.8, 614.3, 617.5, 591801, 96087167],
    [13, 918.5, 1.1, 916.4, 920.7, 726147, 79054712],
    [14, 1318.0, 1.5, 1315.1, 1320.8, 809014, 61384182],
    [15, 1733.2, 1.9, 1729.4, 1737.0, 798681, 46080431],
    [16, 2185.7, 2.6, 2180.6, 2190.7, 721187, 32996192],
    [17, 2505.2, 3.4, 2498.6, 2511.9, 552501, 22053812],
]

female_incidence = np.asarray([row[1] for row in female_incidence_table])
female_incidence_error = np.asarray([row[2] for row in female_incidence_table])

male_incidence = np.asarray([row[1] for row in male_incidence_table])
male_incidence_error = np.asarray([row[2] for row in male_incidence_table])

len(incidence_ages), len(female_incidence), len(male_incidence)

(13, 13, 13)

In [6]:
def logistic_suppression(age, k, t0):
    """
    Late-age logistic suppression term.
    """
    return 1 / (1 + np.exp(-k * (age - t0)))


def incidence_model(age_combined, A, gamma, D0, k, t0, toff=5):
    """
    Cancer incidence model.

    The input age_combined concatenates female ages followed by male ages.
    Sex is not an explicit model variable. Sex-specific predictions enter only
    through the sex-specific TCR diversity curves.
    """
    n = int(len(age_combined) / 2)

    female_age = age_combined[:n]
    male_age = age_combined[n:]

    female_D = female_diversity_spline(female_age - toff)
    male_D = male_diversity_spline(male_age - toff)

    D = np.concatenate([female_D, male_D])
    age = age_combined

    immune_component = np.exp(-D / D0)
    mutation_component = (age - toff) ** gamma
    suppression_component = logistic_suppression(age, k, t0)

    return A * mutation_component * immune_component * suppression_component


parameter_bounds = (
    [2e-9, 0.24, 1e4, -1, 60],
    [100, 20, 1e6, 1, 100],
)

initial_guess = [3.68832163e-01, 2.46547612e00, 1.06161880e05, -0.1, 70]

fit_parameters, fit_covariance = curve_fit(
    incidence_model,
    np.concatenate([incidence_ages, incidence_ages]),
    np.concatenate([female_incidence, male_incidence]),
    sigma=np.concatenate([female_incidence_error, male_incidence_error]),
    p0=initial_guess,
    bounds=parameter_bounds,
    maxfev=10000,
)

fit_parameters

array([ 2.52240857e+00,  2.21976447e+00,  8.99025409e+04, -1.98277160e-01,
        8.23178760e+01])

In [7]:
def incidence_model_with_diversity_shift(
    age_combined,
    A,
    gamma,
    D0,
    k,
    t0,
    diversity_shift=0,
    toff=5,
):
    """
    Evaluate the incidence model after shifting the interpolated TCR diversity
    curves by diversity_shift times their bootstrap uncertainty.

    diversity_shift = +1 means D + 1 sigma.
    diversity_shift = -1 means D - 1 sigma.
    """
    n = int(len(age_combined) / 2)

    female_age = age_combined[:n]
    male_age = age_combined[n:]

    female_D = (
        female_diversity_spline(female_age - toff)
        + diversity_shift * female_diversity_error_spline(female_age - toff)
    )

    male_D = (
        male_diversity_spline(male_age - toff)
        + diversity_shift * male_diversity_error_spline(male_age - toff)
    )

    D = np.concatenate([female_D, male_D])
    age = age_combined

    immune_component = np.exp(-D / D0)
    mutation_component = (age - toff) ** gamma
    suppression_component = logistic_suppression(age, k, t0)

    return A * mutation_component * immune_component * suppression_component


age_model = np.linspace(incidence_ages[0], incidence_ages[-1], 100)
age_model_combined = np.concatenate([age_model, age_model])

model_fit = incidence_model(age_model_combined, *fit_parameters)
female_fit = model_fit[: len(age_model)]
male_fit = model_fit[len(age_model):]

# Use 2 sigma to match the 95% confidence interval language in the manuscript.
ci_nsigma = 2

model_upper = incidence_model_with_diversity_shift(
    age_model_combined,
    *fit_parameters,
    diversity_shift=ci_nsigma,
)

model_lower = incidence_model_with_diversity_shift(
    age_model_combined,
    *fit_parameters,
    diversity_shift=-ci_nsigma,
)

female_fit_upper = model_upper[: len(age_model)]
male_fit_upper = model_upper[len(age_model):]

female_fit_lower = model_lower[: len(age_model)]
male_fit_lower = model_lower[len(age_model):]

In [8]:
# Use all subjects to estimate the age-dependent distribution of log TCR diversity.
all_diversity_log = binned_summary(
    df["age"],
    df["log_tcr_diversity"],
    bins=age_bins,
    interval=(0.16, 0.84),
    n_bootstrap=1001,
    rng=rng,
)

# Estimate intrinsic 1-sigma scatter in log10(D) after subtracting
# measurement uncertainty from repeat samples.
measurement_sigma_logD = 0.10

observed_sigma_logD = (
    all_diversity_log["interval_high"] - all_diversity_log["interval_low"]
) / 2

intrinsic_sigma_logD = np.sqrt(
    observed_sigma_logD**2 - measurement_sigma_logD**2
)

# Use bins with enough age coverage for interpolation in the risk map.
age_knots = all_diversity_log["age_median"].values[2:]
median_logD = all_diversity_log["value_median"].values[2:]
sigma_logD = intrinsic_sigma_logD.values[2:]

percentile_knots = np.linspace(0.05, 0.95, 19)

logD_by_age_and_percentile = np.asarray(
    [
        [
            stats.norm.ppf(p, loc=mu, scale=sigma)
            for p in percentile_knots
        ]
        for mu, sigma in zip(median_logD, sigma_logD)
    ]
)

logD_surface = RectBivariateSpline(
    age_knots,
    percentile_knots,
    logD_by_age_and_percentile,
)

risk_age_grid = np.linspace(40, 75, 300)
risk_percentile_grid = np.linspace(0.05, 0.95, 301)

age_grid, percentile_grid = np.meshgrid(
    risk_age_grid,
    risk_percentile_grid,
    indexing="ij",
)

A, gamma, D0, k, t0 = fit_parameters

D_grid = 10 ** logD_surface.ev(
    age_grid.ravel(),
    percentile_grid.ravel(),
).reshape(age_grid.shape)

risk_grid = (
    A
    * age_grid**gamma
    * np.exp(-D_grid / D0)
    * logistic_suppression(age_grid, k, t0)
)

# Normalize to risk of a 40-year-old subject with median TCR diversity.
D_reference = 10 ** logD_surface.ev(40, 0.5)
risk_reference = (
    A
    * 40**gamma
    * np.exp(-D_reference / D0)
    * logistic_suppression(40, k, t0)
)

relative_risk_grid = risk_grid / risk_reference

relative_risk_grid.shape

(300, 301)

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ----------------------------
# Panel A: incidence model fit
# ----------------------------
ax = axes[0]

ax.fill_between(
    age_model,
    male_fit_lower,
    male_fit_upper,
    alpha=0.2,
    color=colors[0],
)

ax.plot(
    incidence_ages,
    male_incidence,
    color=colors[0],
    marker=".",
    markersize=15,
    linestyle="",
    label="Male Cancer Incidence",
)

ax.plot(
    age_model,
    male_fit,
    color=colors[0],
    linestyle="--",
    linewidth=3,
    label="Male Model Fit",
)

ax.fill_between(
    age_model,
    female_fit_lower,
    female_fit_upper,
    alpha=0.2,
    color=colors[1],
)

ax.plot(
    incidence_ages,
    female_incidence,
    color=colors[1],
    marker="v",
    markersize=9,
    linestyle="",
    label="Female Cancer Incidence",
)

ax.plot(
    age_model,
    female_fit,
    color=colors[1],
    linestyle="--",
    linewidth=3,
    label="Female Model Fit",
)

ax.set_xlim(20, 85)
ax.set_xlabel("Age")
ax.set_ylabel("Cancer Incidence \n[per 100,000 person-years]")
ax.legend(fontsize=12, loc="upper left")

ax.text(
    0.04,
    0.94,
    "(A)",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top",
)

# ----------------------------
# Panel B: relative-risk heatmap
# ----------------------------
ax = axes[1]

heatmap = ax.imshow(
    np.log10(relative_risk_grid),
    aspect="auto",
    extent=[
        risk_percentile_grid.min(),
        risk_percentile_grid.max(),
        risk_age_grid.min(),
        risk_age_grid.max(),
    ],
    origin="lower",
    cmap="plasma",
    interpolation="bilinear",
)

cbar = fig.colorbar(
    heatmap,
    ax=ax,
    label="Relative Risk Factor",
)

risk_ticks = [0.5, 1, 2, 4, 8, 16]
cbar.set_ticks([np.log10(t) for t in risk_ticks])
cbar.set_ticklabels([str(t) for t in risk_ticks])

# Smooth only for contours.
smoothed_relative_risk = gaussian_filter(
    relative_risk_grid,
    sigma=3,
)

contours = ax.contour(
    risk_percentile_grid,
    risk_age_grid,
    smoothed_relative_risk,
    levels=[0.5, 1, 2, 4, 8, 16],
    colors="lightgray",
    linewidths=1,
)

def contour_label_format(x):
    return f"{int(x)}" if x == int(x) else f"{x:.1f}"

midpoints = []

for seglist in contours.allsegs:
    if not seglist:
        continue

    contour_segment = seglist[0]

    mid_x, mid_y = np.mean(contour_segment, axis=0)

    midpoints.append((mid_x, mid_y))

ax.clabel(
    contours,
    inline=True,
    fontsize=11,
    fmt=contour_label_format,
    colors="black",
    manual=midpoints,
)

ax.set_xlabel("TCR Diversity Percentile")
ax.set_ylabel("Age")

# Show percentile labels as percentages.
xticks = ax.get_xticks()
ax.set_xticklabels([f"{int(t * 100)}" for t in xticks])

ax.text(
    0.94,
    0.94,
    "(B)",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="right",
    va="top",
)

/var/folders/x5/vzmwmhlx37g6_cjs3b49_t640000gn/T/ipykernel_26935/2585002794.py:151: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{int(t * 100)}" for t in xticks])


Text(0.94, 0.94, '(B)')

<Figure size 1500x600 with 3 Axes>